# Try `implanet` — a guided tour

Render a planet's equirectangular map into a 2-D view from any direction, add
scientific overlays, make a flat map, pull real Sun geometry from SPICE, and
reproduce a spacecraft flyby.

**Requirements**

```bash
pip install implanet[all]      # core + spiceypy + matplotlib
pip install jupyter            # to run this notebook
```

**Downloads.** Nothing is bundled. Textures and SPICE kernels download lazily
on first use into a cache (`~/.cache/implanet` by default; set `IMPLANET_CACHE`
to relocate, or run inside a repo checkout to reuse `maps/data` & `kernels`).
This notebook pulls roughly: Mars map ~15 MB, generic kernels ~32 MB,
MESSENGER trajectory ~20 MB.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import implanet
from implanet import (
    render_disk, render_flatmap,
    graticule_segments, limb_circle, terminator_segments, subobserver_point,
)
from implanet.assets import get_texture, get_kernel, kernel_entries, texture_entries

print('implanet', implanet.__version__)
print(len(texture_entries()), 'textures /', len(kernel_entries()), 'kernels in the registries')

## 1. Render a disk

`get_texture` returns a local path (downloading if needed). `render_disk`
returns **`(image, x, y)`**: a uint8 array (row 0 = top) plus pixel-edge
coordinates in planet radii — `x` increases left→right, `y` decreases
top→bottom.

`view_direction` points **camera → planet center**; `sun_direction` points
**planet → Sun** (both in the body-fixed IAU frame: +Z = north pole,
+X = prime meridian, +Y = 90°E).

In [ ]:
mars_path = get_texture('Mars')              # downloads on first run
mars = Image.open(mars_path).convert('RGB')

img, x, y = render_disk(
    mars,
    view_direction=(-1, -0.2, -0.3),         # camera -> center
    sun_direction=(1, 0.5, 0.4),             # planet -> Sun
    size=600,
)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img, extent=(x.min(), x.max(), y.min(), y.max()), origin='upper')
ax.set_xlabel('x  [planet radii]'); ax.set_ylabel('y  [planet radii]')
ax.set_title('Mars — render_disk'); plt.show()

## 2. Plot it yourself with `pcolormesh`

The `x`/`y` edges are made for `pcolormesh`'s default `shading='flat'`
(length `W+1` / `H+1`). `pcolormesh` needs scalar data, so render a
single-channel (grayscale) texture and pass it straight through.

In [ ]:
gray = np.asarray(Image.open(mars_path).convert('L'))   # 2-D texture
g, gx, gy = render_disk(gray, view_direction=(-1, -0.2, -0.3),
                        sun_direction=(1, 0.5, 0.4), size=400)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pcolormesh(gx, gy, g, cmap='gray', shading='flat')
ax.set_aspect('equal'); ax.set_title('Mars — pcolormesh from render_disk edges')
plt.show()

## 3. Vector overlays

Overlays come back as parallel `(xs, ys)` lists of 1-D arrays (or a single
`(x, y)` for the limb), so you draw them yourself with `ax.plot`.

In [ ]:
view = (-1, -0.2, -0.3)
sun = (1, 0.4, 0.2)
img, x, y = render_disk(mars, view_direction=view, sun_direction=sun, size=600)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.imshow(img, extent=(x.min(), x.max(), y.min(), y.max()), origin='upper')

g = graticule_segments(view_direction=view)
for grp in (g['parallels'], g['meridians']):
    for xx, yy in zip(*grp):
        ax.plot(xx, yy, '0.6', lw=0.6, ls='--')

lx, ly = limb_circle()
ax.plot(lx, ly, 'k', lw=1.0)

txs, tys = terminator_segments(view, sun)
for xx, yy in zip(txs, tys):
    ax.plot(xx, yy, 'w--', lw=1.4)

ax.plot(0, 0, 'r+', ms=10)                    # sub-observer is always (0, 0)
lat0, lon0 = subobserver_point(view)
ax.set_title(f'overlays — sub-observer ({lat0:+.1f}°, {lon0:+.1f}°)')
ax.set_aspect('equal'); plt.show()

## 4. One call: a publication figure

`plot_planet` composes the disk + dashed graticule + limb + terminator +
sub-observer marker + ticks in planet radii + captions.

In [ ]:
from implanet.figure import plot_planet

fig, ax = plot_planet(
    mars,
    view_direction=(-1, -0.2, -0.3),
    sun_direction=(1, 0.5, 0.4),
    body_name='Mars',
    source_text='Solar System Scope (CC BY 4.0)',
)
plt.show()

## 5. Flat map with rotation & shadow

`render_flatmap` re-shades the equirectangular texture in place.
`rotation_lon_deg` shifts the spin phase; `sun_direction` adds the
day/night terminator.

In [ ]:
flat = render_flatmap(
    mars,
    rotation_lon_deg=45,
    sun_direction=(1, 0.3, 0.2),
    ambient=0.05,
    output_size=(512, 1024),
    return_array=True,
)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.imshow(flat, extent=(-180, 180, -90, 90), aspect='equal', origin='upper')
ax.set_xticks(np.arange(-180, 181, 30)); ax.set_yticks(np.arange(-90, 91, 30))
ax.grid(True, ls='--', lw=0.5, color='white', alpha=0.4)
ax.set_xlabel('display longitude (°)'); ax.set_ylabel('latitude (°)')
ax.set_title('Mars flat map — rotation 45° + shadow'); plt.show()

## 6. Real Sun geometry from SPICE (optional)

Needs `spiceypy`. The first call lazily downloads the ~32 MB generic NAIF
kernels. `sun_direction(body, utc)` returns a body-fixed unit vector you can
feed straight into `render_disk` / `plot_planet`.

In [ ]:
from implanet import sun_direction, sub_solar_point

utc = '2026-05-14T12:00:00'
sun = sun_direction('Mars', utc)
lat, lon = sub_solar_point('Mars', utc)
print('sun (body-fixed):', np.round(sun, 4))
print(f'sub-solar point: ({lat:+.2f}°, {lon:+.2f}°)')

fig, ax = plot_planet(mars, view_direction=(-1, 0, 0), sun_direction=sun,
                      body_name='Mars', title=f'Mars  {utc} UTC')
plt.show()

## 7. Reproduce a flyby: MESSENGER M1 at Mercury

Pull the spacecraft's body-fixed position from its trajectory SPK
(`get_kernel` downloads & caches it), turn it into a view direction, and
render Mercury exactly as MESSENGER saw it on 2008-01-14 20:24 UTC.

In [ ]:
import spiceypy as spice
import implanet.ephemeris as eph

eph.load_kernels()                                   # generic kernels
spice.furnsh(str(get_kernel('messenger_cruise')))    # MESSENGER trajectory

utc = '2008-01-14T20:24:00'
et = spice.str2et(utc)
# Position in J2000 then rotate to IAU_MERCURY (de440s has no Mercury-center
# in the IAU frame chain, so go via pxform — same trick implanet uses).
pos, _ = spice.spkpos('MESSENGER', et, 'J2000', 'LT', 'MERCURY')
R = spice.pxform('J2000', 'IAU_MERCURY', et)
pos = np.asarray(R) @ np.asarray(pos)
dist = np.linalg.norm(pos)
view = tuple(-pos / dist)                             # camera -> center
print(f'distance from Mercury center: {dist:,.0f} km '
      f'(altitude {dist - 2439.7:,.0f} km)')

sun = sun_direction('Mercury', utc)
merc = Image.open(get_texture('Mercury', 'messenger_enhanced_color')).convert('L').convert('RGB')
fig, ax = plot_planet(merc, view_direction=view, sun_direction=sun,
                      ambient=0.04, body_name='Mercury',
                      title=f'MESSENGER M1   {utc} UTC')
plt.show()

## Where things live & next steps

- **Cache dir** — `implanet.assets.cache_base()` / `kernels_dir()` /
  `maps_dir()`. Override with `IMPLANET_CACHE`.
- **Registries** — every texture is in `maps/manifest.json`; every SPICE
  kernel in `maps/kernels.json`. List them with
  `implanet.assets.texture_entries()` / `kernel_entries()`.
- **More flybys** — see the `examples/` scripts: Voyager 1/2, New Horizons,
  Dawn, Galileo (`voyager2_neptune.py`, `new_horizons_pluto.py`,
  `galileo_galilean_moons.py`, …). Each reuses `examples/_flyby.py`.
- **Run the dev test suite** — `pytest tests/` (kept separate from this
  notebook; not shipped in the wheel).

In [ ]:
from implanet.assets import cache_base, kernels_dir, maps_dir
print('cache base :', cache_base())
print('kernels dir:', kernels_dir())
print('maps dir   :', maps_dir())